In [1]:
%tensorflow_version 2.x
import os
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

TensorFlow 2.x selected.


In [2]:
# for colab
from google.colab import drive
drive.mount('/content/drive')
os.chdir("/content/drive/My Drive/Colab Notebooks")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
# that's the file provided in the assignment
# I put it in my Colab Notebooks folder
SEQ_LENGTH = 400 # Max len was found to be 448, by iterating through all sequences, hence clipping it to 400

!python3 prepare_data2.py the-king-james-bible.txt shake [0-9]+:[0-9]+ -m 400

# you can try running the script with the -h flag to get help on the parameters

Split input into 31103 sequences...
Longest sequence is 533 characters. If this seems unreasonable, consider using the maxlen argument!
Removing sequences longer than 400 characters...
31094 sequences remaining.
Longest remaining sequence has length 399.
Removing length-0 sequences...
31094 sequences remaining.
Serialized 100 sequences...
Serialized 200 sequences...
Serialized 300 sequences...
Serialized 400 sequences...
Serialized 500 sequences...
Serialized 600 sequences...
Serialized 700 sequences...
Serialized 800 sequences...
Serialized 900 sequences...
Serialized 1000 sequences...
Serialized 1100 sequences...
Serialized 1200 sequences...
Serialized 1300 sequences...
Serialized 1400 sequences...
Serialized 1500 sequences...
Serialized 1600 sequences...
Serialized 1700 sequences...
Serialized 1800 sequences...
Serialized 1900 sequences...
Serialized 2000 sequences...
Serialized 2100 sequences...
Serialized 2200 sequences...
Serialized 2300 sequences...
Serialized 2400 sequences...


In [30]:
from prepare_data2 import parse_seq
import pickle

# this is just a datasets of "bytes" (not understandable)
data = tf.data.TFRecordDataset("shake.tfrecords")

# this maps a parser function that properly interprets the bytes over the dataset

seq_len = 200
data = data.map(lambda   x:parse_seq(x))
# a map from characters to indices
vocab = pickle.load(open("shake_vocab", mode="rb"))
vocab_size = len(vocab)
# inverse mapping: indices to characters
ind_to_ch = {ind: ch for (ch, ind) in vocab.items()}
char2idx = {u:i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

print(vocab)
print(vocab_size)
print(ind_to_ch)





{'8': 3, 'L': 4, '\ufeff': 5, 'D': 6, '0': 7, '.': 8, '3': 9, ',': 10, '*': 11, 'p': 12, 'e': 13, 's': 14, '-': 15, 'J': 16, 'E': 17, 'C': 18, 'j': 19, 'h': 20, '9': 21, 'q': 22, 'I': 23, 'r': 24, '7': 25, 'a': 26, ' ': 27, ';': 28, 'U': 29, '1': 30, 'K': 31, 'M': 32, 'z': 33, '4': 34, 'b': 35, 'G': 36, 'S': 37, '6': 38, "'": 39, '\n': 40, ':': 41, 'T': 42, 'm': 43, 'Y': 44, 'R': 45, 'Z': 46, 'k': 47, 'A': 48, 'o': 49, 't': 50, 'f': 51, '2': 52, '!': 53, 'Q': 54, 'V': 55, 'l': 56, 'x': 57, '(': 58, 'i': 59, 'v': 60, 'O': 61, 'P': 62, 'B': 63, 'g': 64, 'N': 65, 'y': 66, 'W': 67, 'H': 68, 'w': 69, 'u': 70, 'd': 71, '5': 72, 'F': 73, '?': 74, 'c': 75, 'n': 76, ')': 77, '<PAD>': 0, '<S>': 1, '</S>': 2}
78
{3: '8', 4: 'L', 5: '\ufeff', 6: 'D', 7: '0', 8: '.', 9: '3', 10: ',', 11: '*', 12: 'p', 13: 'e', 14: 's', 15: '-', 16: 'J', 17: 'E', 18: 'C', 19: 'j', 20: 'h', 21: '9', 22: 'q', 23: 'I', 24: 'r', 25: '7', 26: 'a', 27: ' ', 28: ';', 29: 'U', 30: '1', 31: 'K', 32: 'M', 33: 'z', 34: '4', 35

In [31]:
# don't run this cell lol
# it prints all of the data forever
i = 0
maxlen = 0
for elem in data:
    if(i > 5):
      break
    
    i += 1
    # print(elem);
    as_chars = [ind_to_ch[ind] for ind in elem.numpy()]

    # to_chars = "".join(ind_to_ch.get(thing.numpy()[ind for ind in range(len(thing))]) )
    # print(as_chars, "\n")
    print("".join(as_chars), "\n")
#     if(maxlen < len(as_chars)):
#        maxlen = len(as_chars)
# print("Max length of sequence:" + str(maxlen))      

<S>﻿The First Book of Moses:  Called Genesis


</S> 

<S> In the beginning God created the heaven and the earth.

</S> 

<S> And the earth was without form, and void; and darkness was upon
the face of the deep. And the Spirit of God moved upon the face of the
waters.

</S> 

<S> And God said, Let there be light: and there was light.

</S> 

<S> And God saw the light, that it was good: and God divided the light
from the darkness.

</S> 

<S> And God called the light Day, and the darkness he called Night.
And the evening and the morning were the first day.

</S> 



In [0]:
def split_input_target(chunk):
    input_text = chunk[:,:-1] 
    target_text = chunk[:,1:]
    return input_text, target_text

In [33]:
# remember, for training we should batch data etc.
train_data = data.shuffle(30000).padded_batch(128, padded_shapes=[SEQ_LENGTH],drop_remainder=True).repeat()
train_data = train_data.map(split_input_target)

train_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()
# # each batch is shape batch_size x seq_len x vocab_size

i = 0
for batch in train_data:
  i += 1

  if(i>5):
    break
  print("batch")  
  print(batch)
  oh_batch = tf.one_hot(batch, depth=vocab_size)
  # print(oh_batch.shape)
  print("oh_batch")  

  print(oh_batch)


batch
(<tf.Tensor: id=1572140, shape=(128, 399), dtype=int32, numpy=
array([[ 1, 27, 67, ...,  0,  0,  0],
       [ 1, 27, 42, ...,  0,  0,  0],
       [ 1, 27, 48, ...,  0,  0,  0],
       ...,
       [ 1, 27, 23, ...,  0,  0,  0],
       [ 1, 27, 61, ...,  0,  0,  0],
       [ 1, 27, 42, ...,  0,  0,  0]], dtype=int32)>, <tf.Tensor: id=1572141, shape=(128, 399), dtype=int32, numpy=
array([[27, 67, 20, ...,  0,  0,  0],
       [27, 42, 20, ...,  0,  0,  0],
       [27, 48, 76, ...,  0,  0,  0],
       ...,
       [27, 23, 27, ...,  0,  0,  0],
       [27, 61, 27, ...,  0,  0,  0],
       [27, 42, 20, ...,  0,  0,  0]], dtype=int32)>)
oh_batch
tf.Tensor(
[[[[0. 1. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [1. 0. 0. ... 0. 0. 0.]
   [1. 0. 0. ... 0. 0. 0.]
   [1. 0. 0. ... 0. 0. 0.]]

  [[0. 1. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [1. 0. 0. ... 0. 0. 0.]
   [1. 0. 0. ... 0. 0. 0.]
   [1. 0. 0. ... 0. 0. 0

In [34]:
for input_text, target_text in  train_data.take(1):
  print(input_text)
  as_chars = [ind_to_ch[ind] for ind in input_text[3].numpy()]
  print ("Input data: "+ "".join(as_chars), "\n")
  as_chars = [ind_to_ch[ind] for ind in target_text[3].numpy()]
  print ("Target data: "+ "".join(as_chars), "\n")

tf.Tensor(
[[ 1 27 48 ...  0  0  0]
 [ 1 27 48 ...  0  0  0]
 [ 1 27 48 ...  0  0  0]
 ...
 [ 1 27 48 ...  0  0  0]
 [ 1 27 32 ...  0  0  0]
 [ 1 27 67 ...  0  0  0]], shape=(128, 399), dtype=int32)
Input data: <S> Break thou the arm of the wicked and the evil man: seek out his
wickedness till thou find none.

</S><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD><PAD

Note: Input always starts with space?

In [35]:
model = tf.keras.Sequential()
model.add(layers.Embedding(input_dim=78, output_dim=78))

model.add(layers.SimpleRNN(512,
                           return_sequences=True,
                           kernel_initializer = 'glorot_uniform',
                           recurrent_initializer = 'glorot_uniform'))
model.add(layers.Dense(vocab_size))
model.build(tf.TensorShape([128, 399]))
model.summary() 

opt = tf.optimizers.Adam()


Model: "sequential_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_3 (Embedding)      (None, None, 78)          6084      
_________________________________________________________________
simple_rnn_3 (SimpleRNN)     (None, None, 512)         302592    
_________________________________________________________________
dense_3 (Dense)              (None, None, 78)          40014     
Total params: 348,690
Trainable params: 348,690
Non-trainable params: 0
_________________________________________________________________


In [0]:
@tf.function
def run_rnn_on_seq(seq_batch, mask,no_of_elements):# seq_batch has both input and target output
    # print(seq_batch[0].shape) # input shape ((batch_size - 1) X seq_len) : (128,399)
    # print("mask:"+ str(mask))
    seq_mask = tf.sequence_mask(mask, SEQ_LENGTH, dtype=tf.dtypes.float32)#(128,400)
    seq_mask = seq_mask[ : , : -1]
    # print("seq_mask:"+ str(seq_mask)) #(128,399)
    
    #train
    with tf.GradientTape() as tape:
        total_loss = tf.constant(0.)

        logits = model(seq_batch[0])
        # print("logits:"+ str(logits))
        # print("target:"+ str(seq_batch[1]))

        loss_here = tf.nn.sparse_softmax_cross_entropy_with_logits(seq_batch[1], logits)
        
        #apply mask
        loss_here = tf.multiply(seq_mask, loss_here)
        total_loss = tf.reduce_sum(loss_here)
        total_loss /= tf.cast(no_of_elements, tf.float32)
   
    varis = model.trainable_variables
    grads = tape.gradient(total_loss, varis)    
    # this is gradient clipping
    glob_norm = tf.linalg.global_norm(grads)
    grads = [g/glob_norm for g in grads]  
    opt.apply_gradients(zip(grads, varis))

    return total_loss
  

In [0]:
def train(train_data, no_of_batches,EPOCHS,num_generate): 
  for epoch in range(EPOCHS):
    batch_number =0
    for batch in train_data:
      if batch_number > no_of_batches:
        break
     
      batch_number += 1
      # print("batch_number = "+ str(batch_number))
     
      model.reset_states() #To not link 2 batches via hidden states
      mask = [tf.math.count_nonzero(seq) for seq in batch[0]] 
      no_of_elements = tf.reduce_sum(mask)  # non_zero elements
      # print(no_of_elements) # 16715, with padding it would be 128X399=51072
      xent_avg = run_rnn_on_seq(batch,mask,no_of_elements) #sending mask as parameter because iteration in tensor is not possible inside @tf.function

      # if not batch_number % 20:
      #   print("Batch_number: {} Loss: {}".format(batch_number, xent_avg))

      if not batch_number % 50:
        print("Batch_number: {} Loss: {}".format(batch_number, xent_avg))
        generate("King Slayer ",num_generate)
        print()

In [0]:
def generate(start_string, num_generate):

  input_eval = [char2idx[s] for s in start_string]
  input_eval = tf.expand_dims(input_eval, 0)

  text_generated = []

  model.reset_states()
  for i in range(num_generate):
      predictions = model(input_eval)
      # remove the batch dimension
      predictions = tf.squeeze(predictions, 0)
      predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()
      # print("index"+ str(predicted_id))
      # We pass the predicted word as the next input to the model along with the previous hidden state
      input_eval = tf.expand_dims([predicted_id], 0)
      text_generated.append(ind_to_ch[predicted_id])

  print (start_string + ''.join(text_generated))

Question:
model.reset_states() after each epoch or each batch?

In [39]:
train(train_data, 10000,1, 100) #train(train_data, no_of_batches,EPOCHS,num_generate): 


/tensorflow-2.0.0/python3.6/tensorflow_core/python/framework/indexed_slices.py:424: UserWarning: Converting sparse IndexedSlices to a dense Tensor of unknown shape. This may consume a large amount of memory.
  "Converting sparse IndexedSlices to a dense Tensor of unknown shape. "
/tensorflow-2.0.0/python3.6/tensorflow_core/python/framework/indexed_slices.py:424: UserWarning: Converting sparse IndexedSlices to a dense Tensor of unknown shape. This may consume a large amount of memory.
  "Converting sparse IndexedSlices to a dense Tensor of unknown shape. "


Batch_number: 50 Loss: 2.632601261138916
King Slayer aT
KG?</S>bYknIYCI.-wuwuoE4DQRjeC7t?lP*Kax4Glo'fCa(ljheYlWm,kDvYxMQVGRl;.qg!﻿U.B!</S><PAD>l<PAD>gCxuDwIp﻿ZgJ5N﻿5sS

Batch_number: 100 Loss: 2.162553310394287
King Slayer ePhy yvad-p C56<S>: </S>QW0<PAD>ZGhe<S> *-EnLwqAk﻿iI 7f8oRl?.
</S>9gad3MrE, sviwS3me oPFThe
wH-.
ThB6! AHb76D.M0R3

Batch_number: 150 Loss: 1.9989832639694214
King Slayer nCHelca) I of: Ha9CAvege<PAD>M9 F I
LWhMqG3BI Echevedh</S>f I
O.
g(Hu!.
</S>QA4e, I CI JJel'P:, Be6<PAD>: I: wReg'

Batch_number: 200 Loss: 1.8838353157043457
King Slayer e LThVU-h
ThiPN; gy: Hecof.
</S><PAD>WhigG d.
</S><PAD>wUK<PAD>gHerod, om4O E*jund I Wh8th52sCF
O AR) Mof) G.
</S><PAD>y </S>7in

Batch_number: 250 Loss: 1.790482997894287
King Slayer elevean Min Angk.
A J
Vokze? anopre
J</S><PAD>﻿x9'Q
s I SZ: I
Agh's I M, H G.
OQ EdofPaqupup, F) F He Be? K

Batch_number: 300 Loss: 1.6905159950256348
King Slayer exallYerexDaded Anofo9ou(Yeve LO EdQwhe2lllli<PAD>un F G; Gk; S</S><PAD>) Goun, </S><PA

KeyboardInterrupt: ignored